# NB9

Whether MES-tolerance coupling strength tracks neuropathological stage.

In [ ]:
# Stage-dependent coupling

import os, re, gc, glob, time, json, warnings, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.sparse as sp

import matplotlib
matplotlib.rcParams.update({
    "font.family":"Arial","font.size":8,"axes.titlesize":9,"axes.labelsize":8,
    "xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":7,"figure.dpi":150,
    "savefig.dpi":1200,"savefig.bbox":"tight","savefig.pad_inches":0.05,
    "axes.linewidth":0.8,"pdf.fonttype":42,"ps.fonttype":42,
})
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_style("ticks"); HAS_SNS=True
except ImportError:
    HAS_SNS=False
import scanpy as sc
warnings.filterwarnings("ignore"); np.random.seed(42); SEED=42

MIN_CELLS_PER_DONOR = 30   # donors with fewer microglia give unstable coupling -> excluded
MIN_DONORS_FOR_TREND = 6   # need at least this many staged donors to fit a trend

# Paths
BASE_DIR = Path(os.environ.get("MES_BASE_DIR", "."))
RAW_DIR=BASE_DIR/"Raw Data"
PROC_DIR=BASE_DIR/"Process Data"
MANUSCRIPT_DIR=(BASE_DIR/"Manuscript data") if (BASE_DIR/"Manuscript data").exists() else (BASE_DIR/"Manuscript Data")
FIG_DIR=MANUSCRIPT_DIR/"Figures"/"Revision"; FIG_DIR.mkdir(parents=True,exist_ok=True)
TAB_DIR=MANUSCRIPT_DIR/"Tables"/"Revision"; TAB_DIR.mkdir(parents=True,exist_ok=True)
AIM2_DIR=PROC_DIR/"aim2_microglia"

run_log=[]
def log(m):
    s=f"[{time.strftime('%H:%M:%S')}] {m}"; print(s,flush=True); run_log.append({"t":time.strftime('%H:%M:%S'),"m":m})
def save_fig(fig,name):
    out=FIG_DIR/f"Supp_Rev_{name}.png"; fig.savefig(out,dpi=1200,bbox_inches="tight"); plt.close(fig); log(f"FIG saved: {out.name}")
def save_xlsx(sheets,name):
    if isinstance(sheets,pd.DataFrame): sheets={"Sheet1":sheets}
    out=TAB_DIR/f"Supp_Rev_{name}.xlsx"
    with pd.ExcelWriter(out,engine="openpyxl") as w:
        for sn,df in sheets.items():
            (df if (df is not None and len(df)) else pd.DataFrame({"note":["no data"]})).to_excel(w,index=False,sheet_name=str(sn)[:31])
    log(f"TAB saved: {out.name}")
def bh_fdr(p):
    p=np.asarray(p,float); out=np.full_like(p,np.nan); m=np.isfinite(p)
    if m.sum()==0: return out
    pv=p[m]; n=pv.size; o=np.argsort(pv); r=pv[o]; q=r*n/np.arange(1,n+1)
    q=np.minimum.accumulate(q[::-1])[::-1]; om=np.empty_like(pv); om[o]=np.clip(q,0,1); out[m]=om; return out

# Panels
TOL_HOMEO=["P2RY12","CX3CR1","TMEM119","GPR34","SALL1","CSF1R","OLFML3"]
TOL_ACTIV=["APOE","SPP1","LPL","TREM2","CST7","CTSD","TYROBP","FCER1G","LGALS3","CD68"]
MICRO_MARKERS=["P2RY12","CX3CR1","TMEM119","CSF1R","AIF1","TYROBP","LST1","C1QA","C1QB","CTSS"]
MICROGLIA_LABELS={"MG","MIC","MICRO","MICROGLIA","MYELOID","MG1","MG2","MG3","IMMUNE"}

# Pathology stage parsers (reuse NB6 conventions, extend for GSE174367)
def parse_braak(x):
    """Braak / Tangle stage -> ordinal 0..6. Handles 'Stage 6', 'Braak VI', 'VI', '6'."""
    if pd.isna(x): return np.nan
    s=str(x).upper().replace("STAGE","").replace("BRAAK","").replace("TANGLE","").strip()
    # digit first
    m=re.search(r"(\d+)", s)
    if m:
        v=int(m.group(1))
        return float(v) if 0<=v<=6 else np.nan
    # roman numerals (longest first to avoid 'V' matching inside 'VI')
    roman=[("VI",6),("IV",4),("III",3),("II",2),("I",1),("V",5),("0",0)]
    tok=s.replace(".","").strip()
    for r,v in roman:
        if tok==r: return float(v)
    # roman embedded in text: take the first standalone roman token
    for t in re.split(r"[^IVX0-9]+", tok):
        for r,v in roman:
            if t==r: return float(v)
    return np.nan

_CERAD={"ABSENT":0,"NONE":0,"NO":0,"SPARSE":1,"MILD":1,"MODERATE":2,"FREQUENT":3,"SEVERE":3}
def parse_cerad(x):
    """CERAD / Plaque stage -> ordinal 0..3. Handles 'Stage A/B/C', words, numbers."""
    if pd.isna(x): return np.nan
    s=str(x).upper().replace("STAGE","").strip()
    # Plaque stage A/B/C (Thal-like) -> 1/2/3
    abc={"A":1,"B":2,"C":3,"0":0,"NONE":0}
    if s in abc: return abc[s]
    for k,v in _CERAD.items():
        if k in s: return v
    m=re.search(r"(\d+)", s)
    if m:
        v=int(m.group(1))
        return float(v) if 0<=v<=3 else np.nan
    return np.nan

def detect_stage_cols(obs):
    cols=list(map(str,obs.columns)); low={c:c.lower() for c in cols}
    braak=None; cerad=None
    for cand in ["Braak","braak","Braak stage","braak_stage","Tangle.Stage","Tangle_Stage","tangle_stage","Tangle Stage","tangle"]:
        if cand in cols: braak=cand; break
    if braak is None:
        for c in cols:
            if "braak" in low[c] or "tangle" in low[c]: braak=c; break
    for cand in ["CERAD","cerad","CERAD score","cerad_score","Plaque.Stage","Plaque_Stage","plaque_stage","Plaque Stage","plaque","Thal","thal"]:
        if cand in cols: cerad=cand; break
    if cerad is None:
        for c in cols:
            if "cerad" in low[c] or "plaque" in low[c] or "thal" in low[c]: cerad=c; break
    return braak, cerad

# Scoring helpers
def looks_log1p(a,n=2000):
    X=a.X; v=(X.data[:min(X.data.size,n)] if X.data.size else np.array([0.])) if sp.issparse(X) else np.asarray(X).ravel()[:n]
    return (np.nanmax(v)<25) and (np.mean(np.abs(v-np.round(v))>1e-6)>0.2)
def prep(a):
    if "counts" not in a.layers: a.layers["counts"]=a.X.copy()
    if not looks_log1p(a):
        a.X=a.layers["counts"].copy(); sc.pp.normalize_total(a,target_sum=1e4); sc.pp.log1p(a)
def umap(vn): return {str(v).upper():str(v) for v in vn}
def present(a,genes):
    m=umap(a.var_names); out=[]
    for g in genes:
        gg=str(g).upper()
        if gg in m and m[gg] not in out: out.append(m[gg])
    return out
def score(a,genes,name,min_g=3):
    pr=present(a,genes)
    if len(pr)<min_g: a.obs[name]=np.nan; return 0
    try: sc.tl.score_genes(a,pr,score_name=name,use_raw=False); return len(pr)
    except Exception: a.obs[name]=np.nan; return 0
def donor_col_of(obs):
    cols=list(map(str,obs.columns))
    for c in ["donor_id","Donor ID","donor","DonorID","SampleID","Sample.ID","patient_id","subject_id","individual","Donor","Subject"]:
        if c in cols: return c
    low={c:c.lower() for c in cols}
    for c in cols:
        if "donor" in low[c] and ("id" in low[c] or low[c].endswith("donor")): return c
    for c in cols:
        if "sample" in low[c]: return c
    return None

# MES gene sets
log("="*72); log("LOAD: MES gene weights")
def _find_gw():
    for d in ["Manuscript data","Manuscript Data","Manuscript_Data","Manuscript","Process Data"]:
        for td in ["Tables","tables","Table",""]:
            for fn in ["Main_Table1.xlsx","Table1.xlsx"]:
                p=(BASE_DIR/d/td/fn) if td else (BASE_DIR/d/fn)
                if p.exists():
                    try:
                        df=pd.read_excel(p,sheet_name="GeneWeights")
                        if any(str(c).startswith("MES") for c in df.columns): return p,df
                    except Exception: pass
    for p in BASE_DIR.rglob("*Table1*.xlsx"):
        try:
            if "GeneWeights" in pd.ExcelFile(p).sheet_names:
                df=pd.read_excel(p,sheet_name="GeneWeights")
                if any(str(c).startswith("MES") for c in df.columns): return p,df
        except Exception: continue
    return None,None
MAIN_T1,df_weights=_find_gw()
if MAIN_T1 is None: raise FileNotFoundError("Main_Table1.xlsx not found")
log(f"  found: {MAIN_T1}")
mes_cols=[c for c in df_weights.columns if str(c).startswith("MES")]
mes_gene_sets={m: df_weights[["gene",m]].dropna().sort_values(m,ascending=False).head(50)["gene"].astype(str).str.upper().tolist() for m in mes_cols}
log(f"  {len(mes_cols)} modules")

# Build per-cohort cell frames with donor + stage. Reuse scored h5ads where
# present; for GSE174367 load fresh from Raw Data and subset MG.
def load_scored_cohort(path):
    a=sc.read_h5ad(path); a.obs_names_make_unique(); prep(a)
    return a

def load_gse174367():
    folder=None
    for p in RAW_DIR.rglob("GSE174367"):
        if p.is_dir(): folder=p; break
    if folder is None: return None
    h5=list(folder.rglob("*filtered_feature*.h5"))
    if not h5: return None
    a=sc.read_10x_h5(str(h5[0])); a.var_names_make_unique()
    a.var_names=pd.Index([str(v).upper() for v in a.var_names]); a.var_names_make_unique()
    meta_f=list(folder.rglob("*cell_meta*.csv*"))
    if not meta_f: return None
    meta=pd.read_csv(meta_f[0])
    bc="Barcode" if "Barcode" in meta.columns else meta.columns[0]
    meta=meta.set_index(meta[bc].astype(str))
    common=a.obs_names.intersection(meta.index)
    if len(common)==0: return None
    a=a[common].copy(); meta=meta.loc[common]
    a.obs["donor"]=meta["SampleID"].astype(str).values if "SampleID" in meta.columns else "one"
    if "Cell.Type" in meta.columns: a.obs["celltype_ext"]=meta["Cell.Type"].astype(str).values
    for cand,outn in [("Tangle.Stage","Tangle.Stage"),("Plaque.Stage","Plaque.Stage"),
                      ("Age","age"),("Sex","sex"),("PMI","PMI"),("Diagnosis","diagnosis")]:
        if cand in meta.columns: a.obs[outn]=meta[cand].values
    prep(a)
    # subset MG via annotation
    if "celltype_ext" in a.obs.columns:
        lab=a.obs["celltype_ext"].astype(str).str.strip().str.upper()
        mg=lab.isin(MICROGLIA_LABELS)|lab.str.contains("MICRO")|lab.str.contains("MYELOID")
        if mg.sum()>=200: a=a[mg.values].copy()
    return a

log("="*72); log("BUILD per-cohort frames with donor + pathology stage")
cohorts={}
# scored cohorts (SEA-AD, Olah, etc.)
for f in sorted(glob.glob(str(AIM2_DIR/"*__microglia_scored.h5ad"))):
    ds=Path(f).name.replace("__microglia_scored.h5ad","")
    try:
        a=load_scored_cohort(f)
        cohorts[ds]=a
        log(f"  loaded scored: {ds} ({a.n_obs:,} cells)")
    except Exception as e:
        log(f"  {ds} load failed: {e}")
# GSE174367 fresh
try:
    g=load_gse174367()
    if g is not None:
        cohorts["GSE174367"]=g
        log(f"  loaded GSE174367 microglia ({g.n_obs:,} cells)")
    else:
        log("  GSE174367 not loaded (not found or no MG)")
except Exception as e:
    log(f"  GSE174367 load failed: {e}")

# Q1. Per-donor coupling strength + stage, per cohort
log("="*72); log("Q1: per-donor MES-tolerance coupling vs pathology stage")
donor_rows=[]
cohort_stage_info=[]
for ds,a in cohorts.items():
    # score tolerance + MES
    score(a,TOL_HOMEO,"_h"); score(a,TOL_ACTIV,"_a")
    a.obs["tolerance_positioning"]=a.obs["_h"]-a.obs["_a"]
    for m in mes_cols:
        if f"{m}_score" not in a.obs.columns: score(a,mes_gene_sets[m],f"{m}_score")
    dc=donor_col_of(a.obs)
    if dc is None or pd.Series(a.obs[dc]).nunique()<MIN_DONORS_FOR_TREND:
        cohort_stage_info.append({"dataset":ds,"status":f"insufficient donors ({0 if dc is None else pd.Series(a.obs[dc]).nunique()})","usable":False})
        log(f"  {ds}: insufficient donor structure, skipping")
        continue
    braak_c, cerad_c = detect_stage_cols(a.obs)
    has_stage = (braak_c is not None) or (cerad_c is not None)
    if not has_stage:
        cohort_stage_info.append({"dataset":ds,"status":"no pathology stage column","usable":False})
        log(f"  {ds}: no Braak/CERAD/Tangle/Plaque column, skipping")
        continue
    log(f"  {ds}: donor={dc}, braak/tangle={braak_c}, cerad/plaque={cerad_c}")
    obs=a.obs.copy()
    obs["_donor"]=obs[dc].astype(str)
    # per-donor coupling for each MES
    for donor, sub in obs.groupby("_donor"):
        n=len(sub)
        if n < MIN_CELLS_PER_DONOR: continue
        tol=pd.to_numeric(sub["tolerance_positioning"],errors="coerce").to_numpy()
        braak_val = parse_braak(sub[braak_c].iloc[0]) if braak_c else np.nan
        cerad_val = parse_cerad(sub[cerad_c].iloc[0]) if cerad_c else np.nan
        row={"dataset":ds,"donor":donor,"n_cells":int(n),
             "braak_stage":braak_val,"cerad_stage":cerad_val}
        for m in mes_cols:
            msc=f"{m}_score"
            if msc not in sub.columns: row[f"coupling_{m}"]=np.nan; continue
            x=pd.to_numeric(sub[msc],errors="coerce").to_numpy()
            mask=np.isfinite(x)&np.isfinite(tol)
            if mask.sum()>=20:
                r,_=stats.spearmanr(x[mask],tol[mask]); row[f"coupling_{m}"]=float(r)
            else:
                row[f"coupling_{m}"]=np.nan
        donor_rows.append(row)
    cohort_stage_info.append({"dataset":ds,"status":"ok","usable":True,
                              "braak_col":braak_c,"cerad_col":cerad_c,
                              "n_donors_staged":obs["_donor"].nunique()})
    del a; gc.collect()

df_donor=pd.DataFrame(donor_rows)
save_xlsx({"per_donor_coupling":df_donor,"cohort_stage_info":pd.DataFrame(cohort_stage_info)},
          "Q1_PerDonor_Coupling_Stage")

# Q2. Regress coupling strength on stage, per cohort per MES, weighted by sqrt(n)
log("="*72); log("Q2: coupling-vs-stage trend (weighted OLS + Spearman backup)")
def weighted_ols_slope(x, y, w):
    m=np.isfinite(x)&np.isfinite(y)&np.isfinite(w)&(w>0)
    if m.sum()<MIN_DONORS_FOR_TREND: return {"slope":np.nan,"se":np.nan,"p":np.nan,"n":int(m.sum())}
    xa,ya,wa=x[m],y[m],w[m]
    if np.nanstd(xa)==0: return {"slope":np.nan,"se":np.nan,"p":np.nan,"n":int(m.sum())}
    X=np.column_stack([np.ones(len(xa)),xa])
    W=np.diag(wa)
    try:
        XtWX=X.T@W@X; XtWy=X.T@W@ya
        beta=np.linalg.solve(XtWX,XtWy)
        resid=ya-X@beta
        dof=max(len(xa)-2,1)
        sigma2=(resid*wa@resid)/dof
        cov=sigma2*np.linalg.inv(XtWX)
        se=np.sqrt(max(cov[1,1],0))
        t=beta[1]/se if se>0 else np.nan
        p=float(2*stats.t.sf(abs(t),dof)) if np.isfinite(t) else np.nan
        return {"slope":float(beta[1]),"se":float(se),"p":p,"n":int(m.sum())}
    except Exception:
        return {"slope":np.nan,"se":np.nan,"p":np.nan,"n":int(m.sum())}

trend_rows=[]
if len(df_donor):
    for ds in df_donor["dataset"].unique():
        sub=df_donor[df_donor["dataset"]==ds]
        w=np.sqrt(sub["n_cells"].to_numpy())
        for stage_name,stage_col in [("braak/tangle","braak_stage"),("cerad/plaque","cerad_stage")]:
            stage=pd.to_numeric(sub[stage_col],errors="coerce").to_numpy()
            if np.isfinite(stage).sum()<MIN_DONORS_FOR_TREND: continue
            for m in mes_cols:
                ccol=f"coupling_{m}"
                if ccol not in sub.columns: continue
                coupling=pd.to_numeric(sub[ccol],errors="coerce").to_numpy()
                res=weighted_ols_slope(stage,coupling,w)
                # nonparametric backup
                mask=np.isfinite(stage)&np.isfinite(coupling)
                if mask.sum()>=MIN_DONORS_FOR_TREND:
                    rho,prho=stats.spearmanr(stage[mask],coupling[mask])
                else:
                    rho,prho=np.nan,np.nan
                trend_rows.append({"dataset":ds,"stage_type":stage_name,"MES":m,
                                   "slope":res["slope"],"slope_se":res["se"],"slope_p":res["p"],
                                   "spearman_rho":float(rho) if np.isfinite(rho) else np.nan,
                                   "spearman_p":float(prho) if np.isfinite(prho) else np.nan,
                                   "n_donors":res["n"],
                                   "direction":("+" if (np.isfinite(res["slope"]) and res["slope"]>0) else ("-" if np.isfinite(res["slope"]) else "na"))})
df_trend=pd.DataFrame(trend_rows)
if len(df_trend):
    df_trend["q_BH_slope"]=np.nan
    for (ds,st),idx in df_trend.groupby(["dataset","stage_type"]).groups.items():
        df_trend.loc[idx,"q_BH_slope"]=bh_fdr(df_trend.loc[idx,"slope_p"].values)
    df_trend["sig_slope_q05"]=df_trend["q_BH_slope"]<0.05
save_xlsx({"coupling_vs_stage_trend":df_trend}, "Q2_Coupling_vs_Stage_Trend")

# Q3. Cross-cohort direction agreement + verdict
log("="*72); log("Q3: cross-cohort consistency + verdict")
try:
    verdict_rows=[]
    if len(df_trend):
        # For each MES x stage_type, do cohorts agree in direction and is it significant anywhere?
        for m in mes_cols:
            for st in df_trend["stage_type"].unique():
                sub=df_trend[(df_trend["MES"]==m)&(df_trend["stage_type"]==st)]
                if len(sub)==0: continue
                dirs=sub["direction"].tolist()
                pos=sum(1 for d in dirs if d=="+"); neg=sum(1 for d in dirs if d=="-")
                n_sig=int(sub["sig_slope_q05"].sum())
                consistent = (pos==len(sub) or neg==len(sub)) and len(sub)>=2
                verdict_rows.append({"MES":m,"stage_type":st,"n_cohorts":len(sub),
                                     "n_positive":pos,"n_negative":neg,
                                     "direction_consistent":consistent,
                                     "n_cohorts_sig":n_sig,
                                     "mean_slope":float(sub["slope"].mean()),
                                     "mean_spearman_rho":float(sub["spearman_rho"].mean())})
    df_verdict=pd.DataFrame(verdict_rows)
    save_xlsx({"stage_verdict":df_verdict}, "Q_VERDICT")

    # Overall call
    if len(df_verdict):
        strong=df_verdict[(df_verdict["direction_consistent"]==True)&(df_verdict["n_cohorts_sig"]>=2)&(df_verdict["n_cohorts"]>=2)]
        any_consistent=df_verdict[(df_verdict["direction_consistent"]==True)&(df_verdict["n_cohorts"]>=2)]
        log("")
        log(f"  Modules with consistent direction AND sig in >=2 cohorts: {len(strong)}")
        if len(strong):
            for _,r in strong.iterrows():
                log(f"    {r['MES']} [{r['stage_type']}]: {r['n_positive']}+/{r['n_negative']}- , "
                    f"{r['n_cohorts_sig']} sig, mean slope={r['mean_slope']:+.4f}")
            log("")
            log("  => STAGE-DEPENDENCE SUPPORTED for the above module(s). Novel, defensible finding:")
            log("     MES-tolerance coupling strengthens/weakens with neuropathology, replicated across cohorts.")
        elif len(any_consistent):
            log(f"  {len(any_consistent)} module(s) show consistent DIRECTION across cohorts but not")
            log("  significant in >=2 cohorts. Suggestive, not conclusive. Report as trend, not claim.")
        else:
            log("  => NO consistent stage-dependence across cohorts.")
            log("     Coupling strength does not track pathology stage. Report as bounded null.")
        # per-cohort detail
        log("")
        log("  Per-cohort significant trends (q<0.05):")
        sig=df_trend[df_trend["sig_slope_q05"]==True] if len(df_trend) else pd.DataFrame()
        if len(sig):
            for _,r in sig.sort_values(["dataset","MES"]).iterrows():
                log(f"    {r['dataset']:14s} {r['MES']} [{r['stage_type']}]: slope={r['slope']:+.4f} "
                    f"q={r['q_BH_slope']:.3g} rho={r['spearman_rho']:+.2f}")
        else:
            log("    none")
except Exception as e:
    log(f"  Q3 FAILED: {e}\n{traceback.format_exc()}")
    df_verdict=pd.DataFrame()

# FIG: coupling strength vs stage, the clearest MES per cohort
log("="*72); log("FIG: coupling vs stage")
try:
    if len(df_donor) and len(df_trend):
        # pick the MES with the most significant trend overall to illustrate
        best=df_trend.dropna(subset=["slope_p"]).sort_values("slope_p")
        if len(best):
            bm=best.iloc[0]["MES"]; bst=best.iloc[0]["stage_type"]
            stage_col="braak_stage" if "braak" in bst else "cerad_stage"
            cohs=[d for d in df_donor["dataset"].unique()
                  if df_donor[(df_donor["dataset"]==d)][stage_col].notna().sum()>=MIN_DONORS_FOR_TREND]
            if cohs:
                fig,axes=plt.subplots(1,len(cohs),figsize=(max(4,3.2*len(cohs)),3.2),squeeze=False)
                for j,d in enumerate(cohs):
                    ax=axes[0][j]; sub=df_donor[df_donor["dataset"]==d]
                    x=pd.to_numeric(sub[stage_col],errors="coerce"); y=pd.to_numeric(sub[f"coupling_{bm}"],errors="coerce")
                    s=np.sqrt(sub["n_cells"])
                    ax.scatter(x,y,s=np.clip(s,5,60),alpha=0.6,edgecolor="black",linewidth=0.3,color="#4393C3")
                    mask=np.isfinite(x)&np.isfinite(y)
                    if mask.sum()>=MIN_DONORS_FOR_TREND:
                        b=np.polyfit(x[mask],y[mask],1); xs=np.linspace(x[mask].min(),x[mask].max(),20)
                        ax.plot(xs,np.polyval(b,xs),"--",color="#D6604D",lw=1.2)
                    ax.axhline(0,color="grey",lw=0.5)
                    ax.set_xlabel(bst); ax.set_ylabel(f"{bm}-tolerance coupling (per donor)")
                    ax.set_title(d)
                    for sp_ in ["top","right"]: ax.spines[sp_].set_visible(False)
                fig.suptitle(f"Per-donor MES-tolerance coupling vs {bst} ({bm})",y=1.02,fontsize=9)
                save_fig(fig,"Q2_Coupling_vs_Stage")
except Exception as e:
    log(f"  FIG FAILED: {e}")

log("="*72); log("MASTER")
save_xlsx({"Index":pd.DataFrame([
    {"analysis":"Q1","deliverable":"Supp_Rev_Q1_PerDonor_Coupling_Stage.xlsx","addresses":"per-donor MES-tolerance coupling + pathology stage"},
    {"analysis":"Q2","deliverable":"Supp_Rev_Q2_Coupling_vs_Stage_Trend.xlsx","addresses":"weighted trend of coupling vs stage, per cohort per MES"},
    {"analysis":"Q3","deliverable":"Supp_Rev_Q_VERDICT.xlsx","addresses":"cross-cohort direction consistency + verdict"},
])}, "MASTER_NB13_Index")
save_xlsx({"run_log":pd.DataFrame(run_log)}, "MASTER_NB13_RunLog")
log("="*72); log("NB13 COMPLETE"); log(f"  Tables: {TAB_DIR}"); log(f"  Figures: {FIG_DIR}")
